In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [6]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [7]:
# Daten laden
train_path = "data-science-competition-dscb-ss-26/train.csv"
test_path = "data-science-competition-dscb-ss-26/test.csv"
sample_submission_path = "data-science-competition-dscb-ss-26/sample_submission.csv"
output_path = "submission.csv"

train = pd.read_csv(train_path, parse_dates=["date"])
test = pd.read_csv(test_path, parse_dates=["date"])
sample_submission = pd.read_csv(sample_submission_path)

# Kalender Features
train["month"] = train["date"].dt.month
train["weekday"] = train["date"].dt.weekday
test["month"] = test["date"].dt.month
test["weekday"] = test["date"].dt.weekday

print("Train:", train.shape, " Test:", test.shape)

Train: (14319, 17)  Test: (2845, 16)


In [8]:
# Feature Liste festlegen
categorical_features = ["station", "wd"]
numeric_features = [
    "pm10", "so2", "no2", "co", "o3",
    "temp", "pres", "dewp", "rain", "wspm",
    "month", "weekday",
]
feature_columns = categorical_features + numeric_features

In [9]:
# Validation Split
validation_start = pd.Timestamp("2016-01-01")
validation_end   = pd.Timestamp("2016-06-30")

fit_mask   = train["date"] < validation_start
valid_mask = (train["date"] >= validation_start) & (train["date"] <= validation_end)

X_fit   = train.loc[fit_mask,   feature_columns]
y_fit   = train.loc[fit_mask,   "pm25"]
X_valid = train.loc[valid_mask, feature_columns]
y_valid = train.loc[valid_mask, "pm25"]

print("X_fit:", X_fit.shape, " X_valid:", X_valid.shape)

X_fit: (12166, 14)  X_valid: (2153, 14)


In [10]:
# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
    ]
)

In [11]:
# Linear Baseline
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()),
])
linear_model.fit(X_fit, y_fit)
linear_valid_pred = linear_model.predict(X_valid)
linear_valid_mse  = mean_squared_error(y_valid, linear_valid_pred)

In [12]:
regularized_linear_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge()),
])

In [13]:
# Regularisierte Suche
regularized_linear_search = RandomizedSearchCV(
    estimator=regularized_linear_pipeline,
    param_distributions=[
        {"regressor": [Ridge()],      "regressor__alpha": [0.01, 0.1, 1.0, 10.0]},
        {"regressor": [Lasso(max_iter=10000)], "regressor__alpha": [0.01, 0.1, 1.0, 10.0]},
        {"regressor": [ElasticNet(max_iter=10000)],
         "regressor__alpha": [0.01, 0.1, 1.0, 10.0],
         "regressor__l1_ratio": [0.2, 0.5, 0.8]},
    ],
    n_iter=25,
    cv=TimeSeriesSplit(n_splits=3),
    scoring="neg_mean_squared_error",
    random_state=42, n_jobs=1,
)
regularized_linear_search.fit(X_fit, y_fit)

c:\Users\Hagen\miniconda3\envs\py312\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 20 is smaller than n_iter=25. Running 20 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...r', Ridge())])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","[{'regressor': [Ridge()], 'regressor__alpha': [0.01, 0.1, ...]}, {'regressor': [Lasso(max_iter=10000)], 'regressor__alpha': [0.01, 0.1, ...]}, ...]"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",25
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation s

In [14]:
regularized_linear_model = regularized_linear_search.best_estimator_
regularized_linear_valid_pred = regularized_linear_model.predict(X_valid)
regularized_linear_valid_mse  = mean_squared_error(y_valid, regularized_linear_valid_pred)
print(f"Regularisiert Validation-MSE: {regularized_linear_valid_mse:.3f}")

X_train_full = train[feature_columns]
y_train_full = train["pm25"]
X_test       = test[feature_columns]

Regularisiert Validation-MSE: 444.989


In [15]:
# Modell suchen
if regularized_linear_valid_mse < linear_valid_mse:
    best_model = regularized_linear_model
else:
    best_model = linear_model

best_model.fit(X_train_full, y_train_full)
test_pred = best_model.predict(X_test)

In [16]:
# Submission erzeugen + speichern
predictions_by_id = pd.Series(test_pred, index=test["id"])

submission = sample_submission[["id"]].copy()
raw_pred = predictions_by_id.reindex(submission["id"]).to_numpy()

# Clipping: PM2.5 kann nicht negativ sein
min_pm25 = float(train["pm25"].min())
submission["pm25"] = np.round(np.clip(raw_pred, a_min=min_pm25, a_max=None), 3)

submission.to_csv(output_path, index=False)
print("Submission geschrieben:", output_path)
print("Shape:", submission.shape)
submission.head()

Submission geschrieben: submission.csv
Shape: (2845, 2)


,id,pm25
0,Aotizhongxin_2016-07-01,3.000
1,Changping_2016-07-01,3.000
2,Dingling_2016-07-01,3.000
3,Dongsi_2016-07-01,6.351
4,Guanyuan_2016-07-01,6.644
